In [8]:
# Imports.

from pathlib import Path
from time import perf_counter, process_time

import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


In [9]:
# Settings.

EMBEDDING_H5 = Path("artifacts/embeddings/experiment_a_image_only_embeddings.h5")

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

SOLVER = "lbfgs"
MAX_ITER = 1000
RANDOM_STATE = 42
BENCHMARK_LAYER = 20
PARALLEL_CONFIGS = [
    {"n_jobs": 1, "inner_threads": None},
    {"n_jobs": 4, "inner_threads": 32},
    {"n_jobs": 8, "inner_threads": 16},
    {"n_jobs": 16, "inner_threads": 8},
]

print(EMBEDDING_H5, EMBEDDING_H5.exists())
print("Primary metric: AUROC. AUPRC is secondary and should be interpreted against positive prevalence.")


artifacts/embeddings/experiment_a_image_only_embeddings.h5 True
Primary metric: AUROC. AUPRC is secondary and should be interpreted against positive prevalence.


In [10]:
# Load metadata and labels.

with h5py.File(EMBEDDING_H5, "r") as h5:
    labels = h5["labels"][:]
    probe_split = h5["probe_split"][:].astype(str)
    print("labels", labels.shape)
    print("probe_split", pd.Series(probe_split).value_counts().to_dict())
    print("medsiglip/global", h5["medsiglip/global"].shape, h5["medsiglip/global"].dtype)
    print("medgemma/projected_image_mean", h5["medgemma/projected_image_mean"].shape, h5["medgemma/projected_image_mean"].dtype)
    print("medgemma/layer_image_mean", h5["medgemma/layer_image_mean"].shape, h5["medgemma/layer_image_mean"].dtype)
    print("medgemma/layer_last_image", h5["medgemma/layer_last_image"].shape, h5["medgemma/layer_last_image"].dtype)

train_idx = np.where(probe_split == "train")[0]
test_idx = np.where(probe_split == "test")[0]

print("train", len(train_idx))
print("test", len(test_idx))


labels (25000, 5)
probe_split {'train': 20000, 'test': 5000}
medsiglip/global (25000, 1152) float32
medgemma/projected_image_mean (25000, 2560) float32
medgemma/layer_image_mean (25000, 34, 2560) float32
medgemma/layer_last_image (25000, 34, 2560) float32
train 20000
test 5000


In [11]:
# Helper functions.

def current_rss_gb():
    status_path = Path("/proc/self/status")
    if not status_path.exists():
        return np.nan
    for line in status_path.read_text().splitlines():
        if line.startswith("VmRSS:"):
            return float(line.split()[1]) / 1024**2
    return np.nan


def load_feature_matrix(dataset_name, layer=None):
    rss_before = current_rss_gb()
    t0 = perf_counter()
    c0 = process_time()
    with h5py.File(EMBEDDING_H5, "r") as h5:
        if layer is None:
            x_train = h5[dataset_name][train_idx].astype(np.float32)
            x_test = h5[dataset_name][test_idx].astype(np.float32)
        else:
            x_train = h5[dataset_name][train_idx, layer, :].astype(np.float32)
            x_test = h5[dataset_name][test_idx, layer, :].astype(np.float32)

    train_finite = np.isfinite(x_train)
    test_finite = np.isfinite(x_test)
    if not train_finite.all() or not test_finite.all():
        layer_text = "" if layer is None else f" layer={layer}"
        raise ValueError(
            f"Non-finite values in {dataset_name}{layer_text}: "
            f"train_bad={int(x_train.size - train_finite.sum())}, "
            f"test_bad={int(x_test.size - test_finite.sum())}. "
            "Rerun benchmark_embeddings.ipynb or regenerate the benchmark H5 with MedGemma features saved as float32."
        )

    return x_train, x_test, {
        "feature_load_wall_sec": perf_counter() - t0,
        "feature_load_cpu_sec": process_time() - c0,
        "feature_load_rss_before_gb": rss_before,
        "feature_load_rss_after_gb": current_rss_gb(),
    }


def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver=SOLVER,
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
        ),
    )


def fit_one_label(x_train, x_test, label_i, inner_threads=None):
    label_name = TARGET_LABELS[label_i]
    y_train = labels[train_idx, label_i]
    y_test = labels[test_idx, label_i]
    model = make_probe()

    rss_before = current_rss_gb()
    t0 = perf_counter()
    c0 = process_time()
    if inner_threads is None:
        model.fit(x_train, y_train)
    else:
        with threadpool_limits(limits=inner_threads):
            model.fit(x_train, y_train)
    train_time = perf_counter() - t0
    cpu_time = process_time() - c0
    rss_after = current_rss_gb()

    scores = model.predict_proba(x_test)[:, 1]
    positive_prevalence = float(y_test.mean())
    auroc = roc_auc_score(y_test, scores) if len(np.unique(y_test)) == 2 else np.nan
    auprc = average_precision_score(y_test, scores)

    return {
        "label": label_name,
        "solver": SOLVER,
        "positive_prevalence": positive_prevalence,
        "train_time_sec": train_time,
        "train_cpu_sec": cpu_time,
        "fit_rss_before_gb": rss_before,
        "fit_rss_after_gb": rss_after,
        "auroc": auroc,
        "auprc": auprc,
        "n_iter": int(model.named_steps["logisticregression"].n_iter_[0]),
    }


def train_five_label_probes(x_train, x_test):
    rows = [fit_one_label(x_train, x_test, label_i) for label_i in range(len(TARGET_LABELS))]
    total_train_time = sum(row["train_time_sec"] for row in rows)
    total_cpu_time = sum(row["train_cpu_sec"] for row in rows)
    return rows, total_train_time, total_cpu_time


def benchmark_feature(feature_name, dataset_name, layer=None):
    x_train, x_test, load_stats = load_feature_matrix(dataset_name, layer=layer)
    print(feature_name, "x_train", x_train.shape, "x_test", x_test.shape, "load_sec", round(load_stats["feature_load_wall_sec"], 2))

    rows, train_time, cpu_time = train_five_label_probes(x_train, x_test)
    for row in rows:
        row["feature"] = feature_name
        row["feature_dim"] = x_train.shape[1]
        row["five_label_train_time_sec"] = train_time
        row["five_label_train_cpu_sec"] = cpu_time
        row.update(load_stats)

    del x_train, x_test
    return pd.DataFrame(rows)



def train_feature_job(job, inner_threads=None):
    x_train, x_test, load_stats = load_feature_matrix(job["dataset_name"], layer=job.get("layer"))
    t0 = perf_counter()
    rows = [fit_one_label(x_train, x_test, label_i, inner_threads=inner_threads) for label_i in range(len(TARGET_LABELS))]
    wall_time = perf_counter() - t0

    result = {
        "feature": job["feature"],
        "feature_dim": x_train.shape[1],
        "five_label_wall_sec": wall_time,
        "mean_positive_prevalence": float(np.mean([row["positive_prevalence"] for row in rows])),
        "mean_auroc": float(np.mean([row["auroc"] for row in rows])),
        "mean_auprc": float(np.mean([row["auprc"] for row in rows])),
        "max_n_iter": int(np.max([row["n_iter"] for row in rows])),
        **load_stats,
    }

    del x_train, x_test
    return result


def benchmark_parallel_jobs(jobs):
    rows = []
    for config in PARALLEL_CONFIGS:
        n_jobs = config["n_jobs"]
        inner_threads = config["inner_threads"]

        t0 = perf_counter()
        if n_jobs == 1:
            job_results = [train_feature_job(job, inner_threads=inner_threads) for job in jobs]
        else:
            job_results = Parallel(n_jobs=n_jobs)(
                delayed(train_feature_job)(job, inner_threads=inner_threads)
                for job in jobs
            )
        wall_time = perf_counter() - t0

        rows.append({
            "n_jobs": n_jobs,
            "inner_threads": inner_threads if inner_threads is not None else "default",
            "num_feature_jobs": len(jobs),
            "wall_time_sec": wall_time,
            "estimated_experiment_a_wall_min": wall_time * (70 / len(jobs)) / 60,
            "mean_auroc": float(np.mean([row["mean_auroc"] for row in job_results])),
            "mean_auprc": float(np.mean([row["mean_auprc"] for row in job_results])),
            "rss_after_gb": current_rss_gb(),
        })

    return pd.DataFrame(rows)


In [12]:
# Benchmark representative feature matrices.

results = []

results.append(benchmark_feature(
    feature_name="medsiglip_global",
    dataset_name="medsiglip/global",
))

results.append(benchmark_feature(
    feature_name="medgemma_projected_image_mean",
    dataset_name="medgemma/projected_image_mean",
))

results.append(benchmark_feature(
    feature_name="medgemma_layer_image_mean_layer20",
    dataset_name="medgemma/layer_image_mean",
    layer=BENCHMARK_LAYER,
))

results.append(benchmark_feature(
    feature_name="medgemma_layer_last_image_layer20",
    dataset_name="medgemma/layer_last_image",
    layer=BENCHMARK_LAYER,
))

benchmark_df = pd.concat(results, ignore_index=True)
display(benchmark_df)


medsiglip_global x_train (20000, 1152) x_test (5000, 1152) load_sec 0.13
medgemma_projected_image_mean x_train (20000, 2560) x_test (5000, 2560) load_sec 0.26
medgemma_layer_image_mean_layer20 x_train (20000, 2560) x_test (5000, 2560) load_sec 0.45
medgemma_layer_last_image_layer20 x_train (20000, 2560) x_test (5000, 2560) load_sec 0.45


,label,solver,positive_prevalence,train_time_sec,train_cpu_sec,fit_rss_before_gb,fit_rss_after_gb,auroc,auprc,n_iter,feature,feature_dim,five_label_train_time_sec,five_label_train_cpu_sec,feature_load_wall_sec,feature_load_cpu_sec,feature_load_rss_before_gb,feature_load_rss_after_gb
0,Atelectasis,lbfgs,0.3044,1.886799,212.671788,0.691288,0.691368,0.668761,0.440948,862,medsiglip_global,1152,8.307325,939.909419,0.129941,0.131492,0.580254,0.691288
1,Cardiomegaly,lbfgs,0.2196,1.573562,177.681317,0.691368,0.691368,0.811043,0.561185,703,medsiglip_global,1152,8.307325,939.909419,0.129941,0.131492,0.580254,0.691288
2,Consolidation,lbfgs,0.1752,1.644457,187.064720,0.691368,0.691368,0.669898,0.282125,746,medsiglip_global,1152,8.307325,939.909419,0.129941,0.131492,0.580254,0.691288
3,Edema,lbfgs,0.2754,1.640019,186.140658,0.691368,0.691368,0.795914,0.568626,724,medsiglip_global,1152,8.307325,939.909419,0.129941,0.131492,0.580254,0.691288
4,Pleural Effusion,lbfgs,0.4572,1.562489,176.350936,0.691368,0.691368,0.849355,0.824694,680,medsiglip_global,1152,8.307325,939.909419,0.129941,0.131492,0.580254,0.691288
5,Atelectasis,lbfgs,0.3044,3.725486,414.890167,0.807602,0.807602,0.662656,0.435345,529,medgemma_projected_image_mean,2560,18.389347,2055.853231,0.260595,12.406982,0.557377,0.855286
6,Cardiomegaly,lbfgs,0.2196,3.452001,383.027352,0.807602,0.807602,0.805527,0.545646,483,medgemma_projected_image_mean,2560,18.389347,2055.853231,0.260595,12.406982,0.557377,0.855286
7,Consolidation,lbfgs,0.1752,3.773255,423.873308,0.807602,0.807602,0.664184,0.283323,534,medgemma_projected_image_mean,2560,18.389347,2055.853231,0.260595,12.406982,0.557377,0.855286
8,Edema,lbfgs,0.2754,3.840158,432.519387,0.807602,0.807915,0.795377,0.570550,553,medgemma_projected_image_mean,2560,18.389347,2055.853231,0.260595,12.406982,0.557377,0.855286
9,Pleural Effusion,lbfgs,0.4572,3.598448,401.543017,0.807915,0.807915,0.845613,0.821167,486,medgemma_projected_image_mean,2560,18.389347,2055.853231,0.260595,12.406982,0.557377,0.855286


In [13]:
# Summarize probe speed and estimate full Experiment A runtime.

summary = (
    benchmark_df
    .groupby(["feature", "feature_dim"], as_index=False)
    .agg(
        feature_load_wall_sec=("feature_load_wall_sec", "first"),
        feature_load_rss_after_gb=("feature_load_rss_after_gb", "first"),
        five_label_train_time_sec=("five_label_train_time_sec", "first"),
        mean_label_train_time_sec=("train_time_sec", "mean"),
        max_fit_rss_after_gb=("fit_rss_after_gb", "max"),
        mean_positive_prevalence=("positive_prevalence", "mean"),
        mean_auroc=("auroc", "mean"),
        mean_auprc=("auprc", "mean"),
        max_n_iter=("n_iter", "max"),
    )
)

display(summary)

medsiglip_time = summary.query("feature == 'medsiglip_global'")["five_label_train_time_sec"].iloc[0]
projected_time = summary.query("feature == 'medgemma_projected_image_mean'")["five_label_train_time_sec"].iloc[0]
layer_mean_time = summary.query("feature == 'medgemma_layer_image_mean_layer20'")["five_label_train_time_sec"].iloc[0]
layer_last_time = summary.query("feature == 'medgemma_layer_last_image_layer20'")["five_label_train_time_sec"].iloc[0]

estimated_train_time = medsiglip_time + projected_time + 34 * layer_mean_time + 34 * layer_last_time
estimate_df = pd.DataFrame([{
    "solver": SOLVER,
    "estimated_experiment_a_train_time_sec": estimated_train_time,
    "estimated_experiment_a_train_time_min": estimated_train_time / 60,
}])
display(estimate_df)


,feature,feature_dim,feature_load_wall_sec,feature_load_rss_after_gb,five_label_train_time_sec,mean_label_train_time_sec,max_fit_rss_after_gb,mean_positive_prevalence,mean_auroc,mean_auprc,max_n_iter
0,medgemma_layer_image_mean_layer20,2560,0.451678,0.855534,17.973055,3.594611,0.807915,0.28636,0.721556,0.492242,551
1,medgemma_layer_last_image_layer20,2560,0.451008,0.855461,14.807066,2.961413,0.807915,0.28636,0.668949,0.431586,427
2,medgemma_projected_image_mean,2560,0.260595,0.855286,18.389347,3.677869,0.807915,0.28636,0.754671,0.531206,553
3,medsiglip_global,1152,0.129941,0.691288,8.307325,1.661465,0.691368,0.28636,0.758994,0.535516,862


,solver,estimated_experiment_a_train_time_sec,estimated_experiment_a_train_time_min
0,lbfgs,1141.220787,19.020346


In [14]:
# Benchmark process parallelism across feature/layer jobs.

parallel_jobs = [
    {"feature": "medsiglip_global", "dataset_name": "medsiglip/global"},
    {"feature": "medgemma_projected_image_mean", "dataset_name": "medgemma/projected_image_mean"},
    {"feature": "medgemma_layer_image_mean_layer0", "dataset_name": "medgemma/layer_image_mean", "layer": 0},
    {"feature": "medgemma_layer_image_mean_layer10", "dataset_name": "medgemma/layer_image_mean", "layer": 10},
    {"feature": "medgemma_layer_image_mean_layer20", "dataset_name": "medgemma/layer_image_mean", "layer": 20},
    {"feature": "medgemma_layer_image_mean_layer33", "dataset_name": "medgemma/layer_image_mean", "layer": 33},
    {"feature": "medgemma_layer_last_image_layer0", "dataset_name": "medgemma/layer_last_image", "layer": 0},
    {"feature": "medgemma_layer_last_image_layer10", "dataset_name": "medgemma/layer_last_image", "layer": 10},
    {"feature": "medgemma_layer_last_image_layer20", "dataset_name": "medgemma/layer_last_image", "layer": 20},
    {"feature": "medgemma_layer_last_image_layer33", "dataset_name": "medgemma/layer_last_image", "layer": 33},
]

parallel_df = benchmark_parallel_jobs(parallel_jobs)
display(parallel_df)


,n_jobs,inner_threads,num_feature_jobs,wall_time_sec,estimated_experiment_a_wall_min,mean_auroc,mean_auprc,rss_after_gb
0,1,default,10,191.831154,22.380301,0.714651,0.483293,0.571426
1,4,32,10,68.456484,7.986590,0.714641,0.483279,0.572388
2,8,16,10,57.149504,6.667442,0.714638,0.483288,0.572441
3,16,8,10,46.055605,5.373154,0.714588,0.483222,0.572514
